# ETL Transform: News

This notebook runs the **news ETL pipeline**: ingest from Postgres → transform (sentiment, intent, keywords, tickers) → save to `financial_news_transformed` → publish to S3.

**S3 upload modes:**
- **Per-article**: one CSV per article at `news/crypto/year=.../month=.../day=.../hour=.../minute=.../second=.../format=csv/{id}.csv`
- **Batch (run)**: one CSV per run at `news/transformed/crypto/batch=run/format=csv/news_transformed_{timestamp}.csv`
- **Batch (week/month/year)**: one CSV per week, month, or year under `news/transformed/crypto/year=.../week=...` etc.

Set `AWS_NEWS_BUCKET` or `AWS_DEFAULT_BUCKET` in `.env` for S3 uploads.

In [ ]:
import sys
from pathlib import Path

project_root = Path("..").resolve()
src_path = project_root / "src"
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(src_path))

import pandas as pd

In [ ]:
# Run news ETL: transform, save to Postgres, then publish to S3.
# Per-article + batch (run, week, month, year) uploads.

from pipelines.etl_transform import run_news_etl

transformed_df = run_news_etl(
    since="2025-01-01",
    until="2025-12-31",
    news_bucket=None,  # uses AWS_NEWS_BUCKET or AWS_DEFAULT_BUCKET from .env
    save_to_postgres=True,
    upload_s3_per_article=True,
    upload_s3_batch=["run", "week", "month", "year"],
    sentiment_backend="vader",
    extract_tickers=True,
)

print(f"Transformed {len(transformed_df)} articles")

In [ ]:
# Inspect transformed output
if not transformed_df.empty:
    display(transformed_df.head())
    print(transformed_df.columns.tolist())

## Optional: Per-article only (no batch)
Use when you want only one CSV per article in S3.

In [ ]:
# transformed_per_article = run_news_etl(
#     since="2026-01-01",
#     until="2026-01-28",
#     save_to_postgres=True,
#     upload_s3_per_article=True,
#     upload_s3_batch=None,
# )

## Optional: Batch only (no per-article)
Use when you want a single CSV per run, or per week/month/year.

In [ ]:
# transformed_batch = run_news_etl(
#     date="2026-01-27",
#     save_to_postgres=True,
#     upload_s3_per_article=False,
#     upload_s3_batch=["run", "month"],
# )